# Stage 1d — warm-started key-projection training

Stage 1c supported a position-conditioned rank-four key target. This notebook exports that frozen basis and runs four matched seed-one adaptation arms from the same completed CODI seed-one checkpoint. Use an A100 when possible.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/0x0shephard/latent-reasoning.git'
RUN_COMMIT = 'main'  # Replace with the printed SHA before training.
REPO_DIR = '/content/latent-reasoning'
DRIVE_ROOT = '/content/drive/MyDrive/CODI_KAVA'
LOCAL_ROOT = '/content/codikava_runtime'
STAGE1B_STATISTICS = f'{DRIVE_ROOT}/outputs/stage1b_kv_cross_subspaces'
PROJECTION_ARTIFACT = f'{DRIVE_ROOT}/artifacts/stage1d_key_rank4_projectors.pt'
MAX_SECONDS = 32400
EVAL_LIMIT = 200
EXPERIMENTS = [
    'codi_continue_seed1',
    'key_full_seed1',
    'key_rank4_seed1',
    'key_random_rank4_seed1',
]


In [ ]:
import os, pathlib, subprocess, sys

os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception:
    pass

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', RUN_COMMIT], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', os.path.join(REPO_DIR, 'requirements.txt')
], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Checked out:', commit)
if RUN_COMMIT == 'main':
    print('Pin RUN_COMMIT before training:', commit)


In [ ]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Enable a Colab GPU runtime'
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
statistics = Path(STAGE1B_STATISTICS)
if statistics.is_dir():
    statistics = statistics / 'statistics.pt'
warm_start = Path(DRIVE_ROOT) / 'outputs/controls_and_seeds/codi_seed1/checkpoints/step_00096405.pt'
assert statistics.is_file(), statistics
assert warm_start.is_file(), warm_start
checkpoint = torch.load(warm_start, map_location='cpu', weights_only=False, mmap=True)
assert checkpoint.get('step') == 96405
del checkpoint
print('Stage 1b statistics:', statistics)
print('Warm start:', warm_start)


In [ ]:
projection_cmd = [
    sys.executable, 'scripts/export_kv_projection.py',
    '--statistics', STAGE1B_STATISTICS,
    '--output', PROJECTION_ARTIFACT,
    '--rank', '4',
]
subprocess.run(projection_cmd, cwd=REPO_DIR, check=True)
artifact = torch.load(PROJECTION_ARTIFACT, map_location='cpu', weights_only=False)
print({
    'rank': artifact['rank'],
    'shape': tuple(artifact['learned_basis'].shape),
    'processed_examples': artifact['processed_examples'],
    'checkpoint_step': artifact['checkpoint_step'],
    'orthonormality': artifact['maximum_orthonormality_error'],
})


In [ ]:
import datetime, time

logs = Path(DRIVE_ROOT) / 'logs' / 'key_projection'
logs.mkdir(parents=True, exist_ok=True)

def run_persisted(cmd, log_name):
    log_path = logs / log_name
    print('Starting:', ' '.join(cmd), flush=True)
    print('Persistent log:', log_path, flush=True)
    with log_path.open('a', encoding='utf-8', buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(cmd)} ===\n")
        process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
        code = process.wait()
    if code not in (0, 42):
        raise RuntimeError(f'{log_name} failed with exit code {code}')
    return code


In [ ]:
# This is blocking on the Colab VM but restartable from Drive. Closing the browser
# does not guarantee a free Colab runtime will remain allocated.
for experiment in EXPERIMENTS:
    cmd = [
        sys.executable, '-u', 'scripts/colab_key_projection_runner.py',
        '--experiment', experiment,
        '--drive-root', DRIVE_ROOT,
        '--local-root', LOCAL_ROOT,
        '--max-seconds', str(MAX_SECONDS),
        '--eval-limit', str(EVAL_LIMIT),
        '--allow-environment-change',
    ]
    code = run_persisted(cmd, f'{experiment}.log')
    if code == 42:
        print(f'{experiment} needs another session. Rerun this cell; completed arms are restored and skipped by the trainer.')
        break


In [ ]:
import json
from IPython.display import Markdown, display

reports = Path(DRIVE_ROOT) / 'reports'
reports.mkdir(parents=True, exist_ok=True)
report_path = reports / 'stage1d_key_projection_limit200.json'
cmd = [sys.executable, 'scripts/analyze_phase2.py']
for experiment in EXPERIMENTS:
    eval_dir = Path(DRIVE_ROOT) / f'outputs/key_projection/{experiment}/eval/step_00010000'
    summary = eval_dir / 'summary.json'
    assert summary.is_file(), f'Incomplete arm: {experiment}'
    cmd.extend(['--run', f'{experiment}={eval_dir}'])
cmd.extend(['--output', str(report_path)])
subprocess.run(cmd, cwd=REPO_DIR, check=True)
display(Markdown(report_path.with_suffix('.md').read_text()))


In [ ]:
print('\nStage 1d durable outputs')
for experiment in EXPERIMENTS:
    root = Path(DRIVE_ROOT) / f'outputs/key_projection/{experiment}'
    print(f'\n{experiment}')
    for path in sorted(root.rglob('*')):
        if path.is_file():
            print(path.relative_to(root), f'{path.stat().st_size / 1024**2:.1f} MiB')
print('\nReport:', report_path)
